# Reproduce SMITH Figure 3c-f

This notebook regenerates manuscript-matched data panels from real H5AD inputs; it does not read the packaged reference-output tables. [Open the editable source notebook on GitHub](https://github.com/fym0503/SMITH/blob/main/docs/source/tutorials/notebooks/regulatory_section/02_SMITH_Regulatory_Activity_source.ipynb).

## Configure real inputs and fresh outputs

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys
import pandas as pd
from IPython.display import Image, Markdown, display

def find_repository(start):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "reproducibility").exists():
            return candidate
    raise RuntimeError("Run this notebook inside a SMITH repository checkout.")

ROOT = find_repository(Path.cwd().resolve())
DATA_ROOT = Path(os.environ.get("SMITH_TUTORIAL_DATA", "data/tutorials")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("SMITH_TUTORIAL_OUTPUT", "outputs/tutorials")).expanduser().resolve()
CASE_OUTPUT = OUTPUT_ROOT / 'regulatory'
EPOCHS = int(os.environ.get("SMITH_TUTORIAL_EPOCHS", 1))
DEVICE = os.environ.get("SMITH_TUTORIAL_DEVICE", 'cpu')

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


## Verify input files

In [ ]:
inputs = ['regulatory_activity/elegans/splits/elegans_tf/split_1/train.h5ad', 'regulatory_activity/elegans/splits/elegans_tf/split_1/test.h5ad', 'regulatory_activity/elegans/splits/elegans_mirna/split_1/train.h5ad', 'regulatory_activity/elegans/splits/elegans_mirna/split_1/test.h5ad']
rows = []
for relative in inputs:
    path = DATA_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Run scripts/download_tutorial_data.py first.")
    rows.append({"file": relative, "bytes": path.stat().st_size, "sha256": sha256_file(path)})
display(pd.DataFrame(rows))


## Run the workflow for Figure 3c-f

In [ ]:
command = [sys.executable, str(ROOT / 'reproducibility/workflows/regulatory_activity/run_tutorial.py'), "--data-root", str(DATA_ROOT), "--output-dir", str(CASE_OUTPUT), "--device", DEVICE, "--epochs", str(EPOCHS)] + ['--datasets', 'elegans_tf,elegans_mirna', '--splits', 'split_1', '--methods', 'SMITH', '--seeds', '1', '--max-cells', '3000'] + ["--force"]
display_command = ["python", 'reproducibility/workflows/regulatory_activity/run_tutorial.py', "--data-root", "data/tutorials", "--output-dir", "outputs/tutorials/regulatory", "--device", DEVICE, "--epochs", str(EPOCHS)] + ['--datasets', 'elegans_tf,elegans_mirna', '--splits', 'split_1', '--methods', 'SMITH', '--seeds', '1', '--max-cells', '3000'] + ["--force"]
print(" ".join(display_command))
completed = subprocess.run(command, cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
if completed.returncode:
    print(completed.stdout)
    raise subprocess.CalledProcessError(completed.returncode, command)
manifest = json.loads((CASE_OUTPUT / "run_manifest.json").read_text())
print("Generated", manifest["manuscript_figure"], "data from fresh workflow outputs.")


## Inspect newly generated figure data

In [ ]:
for relative in ['figure_data/figure3_c_f_summary.tsv']:
    path = CASE_OUTPUT / relative
    print(relative)
    display(pd.read_csv(path, sep="\t").head(20))


## Draw separate manuscript panels

Every panel below has its own canvas and manuscript-matched aspect ratio. The quick hosted run uses only the methods/repeats executed above; use the full command to regenerate the complete multi-method comparison.

In [ ]:
figure_dir = CASE_OUTPUT / "figures"
plot_command = [sys.executable, str(ROOT / 'reproducibility/workflows/regulatory_activity/plot_figure3.py'), '--values', str(CASE_OUTPUT / 'figure_data/figure3_c_f_values.tsv'), "--output-dir", str(figure_dir)]
subprocess.run(plot_command, cwd=ROOT, check=True)
for heading, relative, width in [('Figure 3c - TF cell-type accuracy', 'figures/figure3_c.png', 430), ('Figure 3d - TF developmental-time correlation', 'figures/figure3_d.png', 430), ('Figure 3e - miRNA cell-type accuracy', 'figures/figure3_e.png', 430), ('Figure 3f - miRNA developmental-time correlation', 'figures/figure3_f.png', 430), ('Shared method legend', 'figures/figure3_method_legend.png', 900)]:
    display(Markdown(f"### {heading}"))
    display(Image(filename=str(CASE_OUTPUT / relative), width=width))
print("Each panel is also exported independently as editable PDF/SVG and 600-dpi TIFF under", figure_dir)


## Full manuscript command

Append the following paper-scale arguments to the workflow command:

```text
--splits split_1,split_2,split_3,split_4,split_5 --methods SMITH,PERSIST-class,PERSIST,ActiveSVM,scGIST,scGeneFit,Spapros --baseline-root external/SMITH_baselines/GPS_tools-main/baselines --baseline-python PERSIST=/opt/envs/persist/bin/python --baseline-python PERSIST-class=/opt/envs/persist/bin/python --baseline-python scGIST=/opt/envs/scgist/bin/python --epochs 200
```

This executed page uses one real TF split, one real miRNA split and SMITH only to keep the hosted example tractable. The paper command above regenerates Figure 3c-f with all five lineage-aware splits and manuscript baselines. Figure 3h-k additionally require versioned module, TF-pair and scRNA-to-TF transfer inputs and are not represented by substitute plots.